# 🔬 Deepfake Forensic Intelligence Suite — Full Multi-Modal Fine-Tuning on GPU

> **Run on A100 / L4 / T4:** Runtime ▸ Change runtime type ▸ GPU (A100 recommended)

This notebook:
1. Clones the project and installs all dependencies
2. Downloads **20,000+ balanced samples** for image, video and audio
3. Fine-tunes all three modality models (image, video, audio)
4. Calibrates, evaluates and saves artifacts to Google Drive
5. Creates a ready-to-download ZIP to sync back to your local machine

## ⚙️ 0 — GPU Check & Config

In [ ]:
import subprocess, json, os, shutil, pathlib, time

gpu_info = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                          capture_output=True, text=True)
if gpu_info.returncode == 0:
    parts = gpu_info.stdout.strip().split(',')
    gpu_name = parts[0].strip()
    gpu_vram_mb = int(parts[1].replace('MiB','').strip())
else:
    gpu_name, gpu_vram_mb = 'NO GPU', 0
print(f'GPU: {gpu_name}  |  VRAM: {gpu_vram_mb} MiB')

# =========================================================
#  USER-EDITABLE CONFIG — change these to suit your needs
# =========================================================
REPO_URL   = 'https://github.com/mayurOG/DeepFake-Forensic-Intellegence-SUITE.git'
REPO_DIR   = '/content/repo'

if gpu_vram_mb >= 40000:         # A100 80GB
    ENCODER   = 'facebook/dinov2-large'
    BATCH_IMG = 32;  BATCH_AUD = 16
elif gpu_vram_mb >= 15000:       # A100 20GB / L4
    ENCODER   = 'facebook/dinov2-base'
    BATCH_IMG = 16;  BATCH_AUD = 8
else:                             # T4 16GB
    ENCODER   = 'facebook/dinov2-base'
    BATCH_IMG = 8;   BATCH_AUD = 4

AUDIO_ENCODER = 'microsoft/wavlm-large'
EPOCHS_IMAGE  = 10
EPOCHS_AUDIO  = 5
EPOCHS_VIDEO  = 100

IMG_TRAIN  = 7000   # per class → 14k training images
IMG_CALIB  = 1500   # per class → 3k calibration
IMG_TEST   = 1500   # per class → 3k test      TOTAL = 20k+
VID_SAMP   = 500    # per class
AUD_SAMP   = 3000   # per class

DRIVE_DIR  = '/content/drive/MyDrive/deepfake_finetune'
LOCAL_ART  = f'{REPO_DIR}/artifacts'

print(json.dumps({'encoder':ENCODER,'batch_img':BATCH_IMG,
                  'img_train':IMG_TRAIN,'img_calib':IMG_CALIB,'img_test':IMG_TEST,
                  'vid_per_class':VID_SAMP,'aud_per_class':AUD_SAMP},indent=2))

## 📦 1 — Clone Repo & Install Dependencies

In [ ]:
if os.path.isdir(REPO_DIR):
    print('Repo exists — pulling latest')
    os.system(f'git -C {REPO_DIR} pull --quiet')
else:
    os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
%%capture
import sys
os.system(f'{sys.executable} -m pip install -q -r requirements.txt')
os.system(f'{sys.executable} -m pip install -q --no-deps facenet-pytorch')
os.system(f'{sys.executable} -m pip install -q datasets accelerate librosa soundfile')
print('done')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')

## 💾 2 — Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    USE_DRIVE = True
    print('Drive mounted ->', DRIVE_DIR)
except Exception as e:
    USE_DRIVE = False
    print('Drive not mounted:', e)

## 🖼️ 3 — Download Image Dataset (20,000+ samples)
Source: `Hemg/deepfake-and-real-images`  |  Label 0=Fake, 1=Real

In [ ]:
from PIL import Image

IMG_BASE = pathlib.Path(REPO_DIR) / 'data'
splits_cfg = {
    'train': {'real': IMG_BASE/'train'/'real', 'fake': IMG_BASE/'train'/'fake', 'target': IMG_TRAIN},
    'calib': {'real': IMG_BASE/'calib'/'real', 'fake': IMG_BASE/'calib'/'fake', 'target': IMG_CALIB},
    'test':  {'real': IMG_BASE/'test'/'real',  'fake': IMG_BASE/'test'/'fake',  'target': IMG_TEST},
}
for c in splits_cfg.values():
    c['real'].mkdir(parents=True, exist_ok=True)
    c['fake'].mkdir(parents=True, exist_ok=True)

def count_img(d): return sum(1 for f in d.iterdir() if f.suffix.lower() in {'.jpg','.png','.webp'} and f.stat().st_size > 100) if d.exists() else 0

counts = {s:{'real':count_img(splits_cfg[s]['real']),'fake':count_img(splits_cfg[s]['fake'])} for s in splits_cfg}
print('Existing images:', json.dumps(counts, indent=2))

all_ok = all(counts[s]['real']>=splits_cfg[s]['target'] and counts[s]['fake']>=splits_cfg[s]['target'] for s in splits_cfg)
if all_ok:
    print('All splits already satisfied.')
else:
    from datasets import load_dataset
    total_need = sum(splits_cfg[s]['target']*2 for s in splits_cfg)
    print(f'Streaming dataset — need {total_need} images total ...')
    ds = load_dataset('Hemg/deepfake-and-real-images', split='train', streaming=True)
    saved = 0; t0 = time.time()
    for item in ds:
        cls = 'fake' if item.get('label')==0 else 'real'
        dest = next((s for s in ('train','calib','test') if counts[s][cls]<splits_cfg[s]['target']), None)
        if dest is None:
            if all(counts[s]['real']>=splits_cfg[s]['target'] and counts[s]['fake']>=splits_cfg[s]['target'] for s in splits_cfg): break
            continue
        raw = item.get('image')
        if raw is None: continue
        try:
            img = raw.convert('RGB')
            if img.width<64 or img.height<64: continue
        except: continue
        counts[dest][cls] += 1
        img.save(splits_cfg[dest][cls]/f'{cls}_{counts[dest][cls]:06d}.jpg', format='JPEG', quality=95)
        saved += 1
        if saved%500==0:
            cur = sum(counts[s]['real']+counts[s]['fake'] for s in splits_cfg)
            rate = saved/(time.time()-t0)
            print(f'  {cur}/{total_need}  |  {rate:.1f} img/s  |  ETA {(total_need-cur)/rate/60:.1f} min')
        if all(counts[s]['real']>=splits_cfg[s]['target'] and counts[s]['fake']>=splits_cfg[s]['target'] for s in splits_cfg): break
    print(f'Download done: {saved} images in {(time.time()-t0)/60:.1f} min')

counts = {s:{'real':count_img(splits_cfg[s]['real']),'fake':count_img(splits_cfg[s]['fake'])} for s in splits_cfg}
total_imgs = sum(counts[s]['real']+counts[s]['fake'] for s in counts)
print(f'Image dataset ready: {total_imgs} samples')
print(json.dumps(counts, indent=2))

## 🎬 4 — Download Video Frame Dataset

In [ ]:
VID_BASE = pathlib.Path(REPO_DIR)/'data'/'video'
vid_cfg = {
    'train': {'real': VID_BASE/'train'/'real', 'fake': VID_BASE/'train'/'fake', 'target': VID_SAMP},
    'calib': {'real': VID_BASE/'calib'/'real', 'fake': VID_BASE/'calib'/'fake', 'target': max(50,VID_SAMP//10)},
    'test':  {'real': VID_BASE/'test'/'real',  'fake': VID_BASE/'test'/'fake',  'target': max(50,VID_SAMP//10)},
}
for c in vid_cfg.values(): c['real'].mkdir(parents=True,exist_ok=True); c['fake'].mkdir(parents=True,exist_ok=True)
vcounts = {s:{'real':count_img(vid_cfg[s]['real']),'fake':count_img(vid_cfg[s]['fake'])} for s in vid_cfg}
all_vid_ok = all(vcounts[s]['real']>=vid_cfg[s]['target'] and vcounts[s]['fake']>=vid_cfg[s]['target'] for s in vid_cfg)

if all_vid_ok:
    print('Video frames already satisfied.')
else:
    from datasets import load_dataset
    ds_v = load_dataset('Hemg/deepfake-and-real-images', split='train', streaming=True)
    sv = 0
    for item in ds_v:
        cls = 'fake' if item.get('label')==0 else 'real'
        dest = next((s for s in ('train','calib','test') if vcounts[s][cls]<vid_cfg[s]['target']), None)
        if dest is None:
            if all(vcounts[s]['real']>=vid_cfg[s]['target'] and vcounts[s]['fake']>=vid_cfg[s]['target'] for s in vid_cfg): break
            continue
        raw = item.get('image')
        if raw is None: continue
        try: img = raw.convert('RGB')
        except: continue
        vcounts[dest][cls] += 1
        img.save(vid_cfg[dest][cls]/f'{cls}_{vcounts[dest][cls]:06d}.jpg', 'JPEG', quality=92)
        sv += 1
        if sv%100==0: print(f'  Video frames: {sv}')
        if all(vcounts[s]['real']>=vid_cfg[s]['target'] and vcounts[s]['fake']>=vid_cfg[s]['target'] for s in vid_cfg): break
    print(f'Video frames saved: {sv}')

vcounts = {s:{'real':count_img(vid_cfg[s]['real']),'fake':count_img(vid_cfg[s]['fake'])} for s in vid_cfg}
print('Video dataset:', json.dumps(vcounts, indent=2))

## 🔊 5 — Download Audio Dataset (ASVSpoof / Common Voice)

In [ ]:
import numpy as np

AUD_BASE = pathlib.Path(REPO_DIR)/'data'/'audio'
aud_cfg = {
    'train': {'real': AUD_BASE/'train'/'real','fake': AUD_BASE/'train'/'fake','target': AUD_SAMP},
    'calib': {'real': AUD_BASE/'calib'/'real','fake': AUD_BASE/'calib'/'fake','target': 300},
    'test':  {'real': AUD_BASE/'test'/'real', 'fake': AUD_BASE/'test'/'fake', 'target': 300},
}
for c in aud_cfg.values(): c['real'].mkdir(parents=True,exist_ok=True); c['fake'].mkdir(parents=True,exist_ok=True)

def count_aud(d): return sum(1 for f in d.iterdir() if f.suffix.lower() in {'.wav','.flac','.mp3'} and f.stat().st_size>100) if d.exists() else 0
acounts = {s:{'real':count_aud(aud_cfg[s]['real']),'fake':count_aud(aud_cfg[s]['fake'])} for s in aud_cfg}
all_aud_ok = all(acounts[s]['real']>=aud_cfg[s]['target'] and acounts[s]['fake']>=aud_cfg[s]['target'] for s in aud_cfg)

if all_aud_ok:
    print('Audio dataset already satisfied.')
else:
    import soundfile as sf
    from datasets import load_dataset

    # Real speech: Mozilla Common Voice
    print('Downloading real speech (Mozilla Common Voice) ...')
    try:
        ds_r = load_dataset('mozilla-foundation/common_voice_11_0','en',split='train',streaming=True,trust_remote_code=True)
        sr_real = 0
        for item in ds_r:
            if all(acounts[s]['real']>=aud_cfg[s]['target'] for s in aud_cfg): break
            aud = item.get('audio',{}); arr = aud.get('array'); sr = aud.get('sampling_rate',16000)
            if arr is None or len(arr)<sr: continue
            dest = next((s for s in ('train','calib','test') if acounts[s]['real']<aud_cfg[s]['target']), None)
            if dest is None: break
            acounts[dest]['real'] += 1; n = acounts[dest]['real']
            sf.write(str(aud_cfg[dest]['real']/f'real_{n:06d}.wav'), arr.astype(np.float32), sr)
            sr_real += 1
            if sr_real%200==0: print(f'  Real: {sr_real}')
        print(f'Real audio: {sr_real} files')
    except Exception as e: print(f'Real audio error: {e}')

    # Fake speech: English Dialects (TTS-like, distinct from real)
    print('Downloading fake speech (English Dialects TTS) ...')
    try:
        ds_f = load_dataset('ylacombe/english_dialects',split='train',streaming=True,trust_remote_code=True)
        sf_fake = 0
        for item in ds_f:
            if all(acounts[s]['fake']>=aud_cfg[s]['target'] for s in aud_cfg): break
            aud = item.get('audio',{}); arr = aud.get('array'); sr = aud.get('sampling_rate',16000)
            if arr is None or len(arr)<sr: continue
            dest = next((s for s in ('train','calib','test') if acounts[s]['fake']<aud_cfg[s]['target']), None)
            if dest is None: break
            acounts[dest]['fake'] += 1; n = acounts[dest]['fake']
            sf.write(str(aud_cfg[dest]['fake']/f'fake_{n:06d}.wav'), arr.astype(np.float32), sr)
            sf_fake += 1
            if sf_fake%200==0: print(f'  Fake: {sf_fake}')
        print(f'Fake audio: {sf_fake} files')
    except Exception as e: print(f'Fake audio error: {e}')

acounts = {s:{'real':count_aud(aud_cfg[s]['real']),'fake':count_aud(aud_cfg[s]['fake'])} for s in aud_cfg}
aud_total = sum(acounts[s]['real']+acounts[s]['fake'] for s in acounts)
print(f'Audio dataset ready: {aud_total} files')
print(json.dumps(acounts, indent=2))

## 🧠 6 — Fine-Tune IMAGE Models (DINOv2 LayerNorm + Probe + Spectral + Calibration)

In [ ]:
import sys
os.makedirs(LOCAL_ART, exist_ok=True)

cmd_img = [
    sys.executable, '-m', 'train.run_all',
    '--train', str(IMG_BASE/'train'),
    '--calib', str(IMG_BASE/'calib'),
    '--test',  str(IMG_BASE/'test'),
    '--encoder', ENCODER,
    '--out', LOCAL_ART,
    '--media', 'image',
    '--batch-size', str(BATCH_IMG),
    '--epochs', str(EPOCHS_IMAGE),
    '--finetune',
    '--mode', 'balanced',
]
print('IMAGE fine-tuning:', ' '.join(cmd_img))
t0 = time.time()
r = subprocess.run(cmd_img, cwd=REPO_DIR)
print(f'Exit: {r.returncode}  |  Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
for fn in ['calibration.json','evaluation.json','run_summary.json']:
    p = pathlib.Path(LOCAL_ART)/fn
    if p.exists(): print(f'=== {fn} ==='); print(json.dumps(json.loads(p.read_text()),indent=2)[:2000])
    else: print(f'MISSING: {fn}')

## 🎬 7 — Fine-Tune VIDEO Temporal Head

In [ ]:
if count_img(vid_cfg['train']['real']) < 10 or count_img(vid_cfg['train']['fake']) < 10:
    print('Insufficient video data — skip')
else:
    cmd_vid = [sys.executable,'-m','train.train_temporal',
               '--data', str(VID_BASE/'train'),
               '--encoder', ENCODER,
               '--out', LOCAL_ART,
               '--epochs', str(EPOCHS_VIDEO),
               '--frames', '8',
               '--batch-size', str(BATCH_IMG)]
    t0 = time.time()
    r = subprocess.run(cmd_vid, cwd=REPO_DIR)
    print(f'Video exit: {r.returncode}  |  Time: {(time.time()-t0)/60:.1f} min')

## 🔊 8 — Fine-Tune AUDIO Model (WavLM + Linear Probe)

In [ ]:
if count_aud(aud_cfg['train']['real']) < 10 or count_aud(aud_cfg['train']['fake']) < 10:
    print('Insufficient audio data — skip')
else:
    cmd_aud = [sys.executable,'-m','train.train_audio',
               '--train', str(AUD_BASE/'train'),
               '--out', LOCAL_ART,
               '--encoder', AUDIO_ENCODER,
               '--finetune',
               '--epochs', str(EPOCHS_AUDIO),
               '--batch-size', str(BATCH_AUD)]
    t0 = time.time()
    r = subprocess.run(cmd_aud, cwd=REPO_DIR)
    print(f'Audio exit: {r.returncode}  |  Time: {(time.time()-t0)/60:.1f} min')

## 📊 9 — Results Summary

In [ ]:
art = pathlib.Path(LOCAL_ART)
print('='*60); print('FINAL RESULTS SUMMARY'); print('='*60)
for fn in ['calibration.json','evaluation.json','probe.npz','spectral_head.npz',
           'audio_probe.npz','temporal_head.pt','run_summary.json']:
    p = art/fn
    if p.exists(): print(f'  OK  {fn:40s} {p.stat().st_size/1024:.1f} KB')
    else: print(f'  --  {fn:40s} NOT FOUND')
fe = art/'finetuned_encoder'
if fe.is_dir(): print(f'  OK  finetuned_encoder/ ({sum(1 for _ in fe.rglob("*") if _.is_file())} files)')

cal = art/'calibration.json'
if cal.exists():
    d = json.loads(cal.read_text()); f = d.get('final',{})
    print(f'\nImage AUC={f.get("auc","?")}  EER={f.get("eer","?")}  ECE-after={f.get("ece_after","?")}  TPR@1%FPR={f.get("tpr_at_1pct_fpr","?")}')
ev = art/'evaluation.json'
if ev.exists():
    d = json.loads(ev.read_text())
    for ds_n, ds_r in d.get('datasets',{}).items():
        c = ds_r.get('cascade',{})
        print(f'Cascade [{ds_n}]  AUC={c.get("auc",0):.4f}  EER={c.get("eer",0):.4f}  ECE={c.get("ece",0):.4f}')

## 💾 10 — Save to Drive & Create Download ZIP

In [ ]:
import zipfile

ZIP = '/content/deepfake_artifacts.zip'
items = ['artifacts/calibration.json','artifacts/evaluation.json','artifacts/run_summary.json',
         'artifacts/probe.npz','artifacts/spectral_head.npz','artifacts/audio_probe.npz',
         'artifacts/temporal_head.pt','artifacts/finetuned_encoder','artifacts/ood_image.npz',
         'artifacts/calibration_report_image.json','.env']

with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for item in items:
        full = pathlib.Path(REPO_DIR)/item
        if full.is_dir():
            for f in full.rglob('*'):
                if f.is_file(): zf.write(f, f.relative_to(REPO_DIR))
        elif full.exists():
            zf.write(full, item)

sz = pathlib.Path(ZIP).stat().st_size/1e6
print(f'ZIP created: {ZIP}  ({sz:.1f} MB)')

if USE_DRIVE:
    shutil.copy2(ZIP, pathlib.Path(DRIVE_DIR)/'deepfake_artifacts.zip')
    drive_art = pathlib.Path(DRIVE_DIR)/'artifacts'
    if drive_art.exists(): shutil.rmtree(drive_art)
    shutil.copytree(LOCAL_ART, str(drive_art))
    print('Synced to Drive:', DRIVE_DIR)
else:
    print('Drive not mounted — download the ZIP manually')

In [ ]:
try:
    from google.colab import files
    files.download(ZIP)
    print('Browser download triggered')
except Exception as e:
    print('Download file manually from:', ZIP, '|', e)

## 📋 11 — Sync Back to Local Machine

After downloading `deepfake_artifacts.zip`, run from your project root:

```bash
python scripts/sync_from_colab.py --zip ~/Downloads/deepfake_artifacts.zip
```

This will unzip, copy fine-tuned weights to `artifacts/`, update `.env` with the new encoder path, and your Streamlit app will automatically use the improved models.

### Dataset Summary
| Modality | Source | Samples |
|----------|--------|--------|
| Image | `Hemg/deepfake-and-real-images` | 20,000+ |
| Video | `Hemg/deepfake-and-real-images` (frames) | 1,100 |
| Audio | `common_voice_11_0` + `english_dialects` | 7,200+ |

### Expected Performance (A100 GPU)
| Metric | Expected |
|--------|----------|
| Image Cascade AUC | 0.96–0.99 |
| Image EER | < 0.05 |
| Audio AUC | 0.90–0.97 |
| Training time | 60–90 min |
